# Graph Neural Networks from Scratch — Tutorial 2: Graph Convolutional Networks from Scratch

### From neighbourhood aggregation to learning meaningful node representations

*🐍 Python · NumPy · PyTorch | Beginner–Intermediate*

---

## Series Roadmap

In Tutorial 1, we explored the fundamental concepts behind graphs: nodes, edges, node features, adjacency matrices, and how information can be aggregated across neighbouring nodes.

Now that we understand how graph data is represented, the next step is to make this information useful for machine learning.

How can a neural network learn from both the features of a node and the structure of the graph? How can neighbouring nodes influence one another, and how can we learn better representations through this process?

This is the idea behind **Graph Convolutional Networks (GCNs)**.

In this tutorial, we will build a GCN from scratch, starting with the adjacency matrix and gradually developing the mathematical operations required for graph convolution.

We'll continue following the philosophy of this series: understand the mathematics first, implement the operations ourselves, and only then move towards higher-level GNN libraries and practical applications.

| # | Tutorial | Focus | Status |
|---|---|---|---|
| 1 | Graph Fundamentals and Representations | nodes, edges, adjacency matrices, graph features | ✅ |
| **2** | **Graph Convolutional Networks from Scratch** | **propagation, normalization, NumPy implementation** | **📍 You are here** |
| 3 | Training a GCN with PyTorch | trainable parameters, backpropagation, node classification | Planned |
| 4 | PyTorch Geometric and the Cora Dataset | practical GNN implementation, real graph dataset | Planned |
| 5 | Message Passing Neural Networks | message, aggregate, update, custom GNN layers | Planned |
| 6 | Graph Attention Networks (GAT) | attention-based neighbourhood aggregation | Planned |
| 7 | GraphSAGE and Advanced Aggregation | scalable neighbourhood learning | Planned |
| 8 | Graph-Level Learning | graph classification and regression | Planned |
| 9 | GNNs for Mesh-Based Systems | mesh connectivity, geometric features, scientific data | Planned |
| 10 | GNNs for Scientific Machine Learning | surrogate modelling and engineering applications | Planned |

The goal is to progress from understanding graph representations to building neural networks that can learn from complex connected systems, including computational meshes and physical systems.

> **New to Graph Neural Networks?** You don't need prior experience with GNN architectures to follow this tutorial. Tutorial 1 introduced the essential graph concepts, and we'll build the GCN step by step. Basic Python and NumPy knowledge, along with a general understanding of neural networks, will be helpful.
>
> If you're interested in scientific machine learning, CFD, or computational mechanics, keep in mind that graph convolution is particularly interesting for data represented on unstructured meshes. Instead of relying on regular grid structures, GNNs can use mesh connectivity to exchange information between neighbouring cells, nodes, or other physical entities.

**Let's now move from representing graphs to learning from them by building our first Graph Convolutional Network.**


---

# 1 — Quick recap from Tutorial 1

In the previous tutorial, we represented a graph using:

- an adjacency matrix $A$
- a feature matrix $X$

For example:

$$
A =
\begin{bmatrix}
0 & 1 & 1 & 0 & 0\\
1 & 0 & 0 & 1 & 0\\
1 & 0 & 0 & 0 & 1\\
0 & 1 & 0 & 0 & 1\\
0 & 0 & 1 & 1 & 0
\end{bmatrix}
$$

and

$$
X =
\begin{bmatrix}
x_1^T\\
x_2^T\\
x_3^T\\
x_4^T\\
x_5^T
\end{bmatrix}.
$$

We discovered that:

$$
AX
$$

performs neighbour aggregation.

For example, the first row of $AX$ is:

$$
x_2+x_3
$$

because node 1 is connected to nodes 2 and 3.

So why isn't this enough?

The answer is **normalization**.



# 2 — Why does a GCN need normalization?

Imagine two nodes.

```text
Node A ── Node B
```

Now imagine a different node with many neighbours:

```text
             Node B
                |
Node C ── Node A ── Node D
                |
             Node E
```

If we simply calculate:

$$
AX
$$

then Node A receives information from four neighbours while Node B receives information from only one.

The magnitude of the aggregated feature therefore depends strongly on the **number of neighbours**.

This can make training unstable and causes high-degree nodes to behave differently from low-degree nodes.

We need a way to account for node degree.

That leads us to the **degree matrix**.



# 3 — The degree of a node

The degree of a node is simply:

> **How many neighbours does this node have?**

For example:

```text
      B
      |
C ─── A ─── D
      |
      E
```

Node A has degree:

$$
d_A = 4
$$

Node B has degree:

$$
d_B = 1
$$

Mathematically:

$$
d_i = \sum_j A_{ij}
$$

In matrix form, we create a diagonal matrix called the **degree matrix**:

$$
D =
\begin{bmatrix}
d_1 & 0 & \cdots & 0\\
0 & d_2 & \cdots & 0\\
\vdots & \vdots & \ddots & \vdots\\
0 & 0 & \cdots & d_N
\end{bmatrix}
$$


In [1]:

import numpy as np

# Recreate the graph from Tutorial 1

nodes = ["Alice", "Bob", "Charlie", "David", "Eve"]

edges = [
    ("Alice", "Bob"),
    ("Alice", "Charlie"),
    ("Bob", "David"),
    ("Charlie", "Eve"),
    ("David", "Eve"),
]

node_to_id = {node: i for i, node in enumerate(nodes)}

A = np.zeros((len(nodes), len(nodes)))

for a, b in edges:
    i = node_to_id[a]
    j = node_to_id[b]
    A[i, j] = 1
    A[j, i] = 1

print("Adjacency matrix:")
print(A)

# Degree of each node
degrees = A.sum(axis=1)

print("\nNode degrees:")
for node, degree in zip(nodes, degrees):
    print(f"{node}: {int(degree)}")


Adjacency matrix:
[[0. 1. 1. 0. 0.]
 [1. 0. 0. 1. 0.]
 [1. 0. 0. 0. 1.]
 [0. 1. 0. 0. 1.]
 [0. 0. 1. 1. 0.]]

Node degrees:
Alice: 2
Bob: 2
Charlie: 2
David: 2
Eve: 2


In [2]:

# Construct the degree matrix

D = np.diag(degrees)

print("Degree matrix D:")
print(D)


Degree matrix D:
[[2. 0. 0. 0. 0.]
 [0. 2. 0. 0. 0.]
 [0. 0. 2. 0. 0.]
 [0. 0. 0. 2. 0.]
 [0. 0. 0. 0. 2.]]



### 🧪 Hands-on Exercise 1

Before continuing, verify the degree matrix manually.

Answer:

1. What is Alice's degree?
2. What is Bob's degree?
3. Which nodes have the same degree?
4. Why is $D$ diagonal?

Then change the graph by adding:

```text
Bob ── Eve
```

and recompute $D$.

The important idea is:

$$
\boxed{D_{ii} = \text{degree of node }i}
$$



# 4 — Adding self-loops

In Tutorial 1, we introduced:

$$
\hat A = A + I
$$

where $I$ is the identity matrix.

Why?

Because otherwise:

$$
AX
$$

contains only information from neighbours.

Adding $I$ means every node also receives its **own features**.

For example:

```text
Before:

Alice ← Bob
Alice ← Charlie


After adding self-loop:

             Alice
            ↙  ↓  ↘
          Bob      Charlie
```

Mathematically:

$$
\hat A = A + I
$$

Now the degree also changes.

If a node originally has degree $d_i$, then after adding a self-loop:

$$
\hat d_i = d_i + 1
$$


In [3]:

I = np.eye(len(nodes))

A_hat = A + I

print("A_hat = A + I:")
print(A_hat)

# New degrees
degrees_hat = A_hat.sum(axis=1)

print("\nDegrees after adding self-loops:")
for node, degree in zip(nodes, degrees_hat):
    print(f"{node}: {int(degree)}")


A_hat = A + I:
[[1. 1. 1. 0. 0.]
 [1. 1. 0. 1. 0.]
 [1. 0. 1. 0. 1.]
 [0. 1. 0. 1. 1.]
 [0. 0. 1. 1. 1.]]

Degrees after adding self-loops:
Alice: 3
Bob: 3
Charlie: 3
David: 3
Eve: 3



# 5 — Why not simply divide by the degree?

One simple idea would be:

$$
D^{-1}A
$$

This normalizes each node's aggregated information by the number of neighbours.

This is closely related to taking an **average** of neighbour features.

For example:

$$
\frac{1}{d_i}
\sum_{j \in \mathcal N(i)} h_j
$$

This is already much better than raw summation.

But the original GCN formulation uses a **symmetric normalization**:

$$
\boxed{
\hat D^{-1/2}
\hat A
\hat D^{-1/2}
}
$$

Why?

It treats the two ends of an edge symmetrically and produces a well-behaved propagation operator for the GCN.

We will construct it explicitly rather than treating it as a mysterious formula.



# 6 — Computing $D^{-1/2}$

Suppose:

$$
D =
\begin{bmatrix}
3 & 0\\
0 & 4
\end{bmatrix}
$$

Then:

$$
D^{-1/2}
=
\begin{bmatrix}
1/\sqrt{3} & 0\\
0 & 1/\sqrt{4}
\end{bmatrix}
$$

In general:

$$
D^{-1/2}_{ii}
=
\frac{1}{\sqrt{d_i}}
$$

Let's implement this.


In [4]:

# D_hat is the degree matrix of A_hat

D_hat = np.diag(degrees_hat)

# Inverse square root of each degree
D_hat_inv_sqrt = np.diag(1.0 / np.sqrt(degrees_hat))

print("D_hat:")
print(D_hat)

print("\nD_hat^(-1/2):")
print(D_hat_inv_sqrt)


D_hat:
[[3. 0. 0. 0. 0.]
 [0. 3. 0. 0. 0.]
 [0. 0. 3. 0. 0.]
 [0. 0. 0. 3. 0.]
 [0. 0. 0. 0. 3.]]

D_hat^(-1/2):
[[0.57735027 0.         0.         0.         0.        ]
 [0.         0.57735027 0.         0.         0.        ]
 [0.         0.         0.57735027 0.         0.        ]
 [0.         0.         0.         0.57735027 0.        ]
 [0.         0.         0.         0.         0.57735027]]



### 🧪 Hands-on Exercise 2

For the first node, calculate manually:

$$
\frac{1}{\sqrt{\hat d_1}}
$$

and compare it with the first diagonal value of `D_hat_inv_sqrt`.

Then answer:

> What happens to the normalization factor as the node degree becomes larger?

Hint: compare:

$$
\frac{1}{\sqrt{1}},
\qquad
\frac{1}{\sqrt{4}},
\qquad
\frac{1}{\sqrt{16}}
$$



# 7 — Constructing the normalized adjacency matrix

Now we have everything we need.

Define:

$$
\boxed{
S =
\hat D^{-1/2}
\hat A
\hat D^{-1/2}
}
$$

We will call $S$ the **normalized adjacency / propagation matrix** in this tutorial.

It determines how information is propagated through the graph while accounting for node degree.


In [5]:

S = D_hat_inv_sqrt @ A_hat @ D_hat_inv_sqrt

np.set_printoptions(precision=3, suppress=True)

print("Normalized adjacency matrix S:")
print(S)


Normalized adjacency matrix S:
[[0.333 0.333 0.333 0.    0.   ]
 [0.333 0.333 0.    0.333 0.   ]
 [0.333 0.    0.333 0.    0.333]
 [0.    0.333 0.    0.333 0.333]
 [0.    0.    0.333 0.333 0.333]]



Take a moment to inspect the matrix.

It is no longer just zeros and ones.

An edge now has a weight determined by the degrees of the nodes it connects:

$$
S_{ij}
=
\frac{\hat A_{ij}}
{\sqrt{\hat d_i\hat d_j}}
$$

for connected nodes.

This means:

> **A message received from a neighbour is scaled according to the connectivity of both nodes.**



# 8 — The GCN layer

We can now write the complete GCN layer:

$$
\boxed{
H^{(l+1)}
=
\sigma
\left(
S H^{(l)} W^{(l)}
\right)
}
$$

where:

$$
S =
\hat D^{-1/2}
\hat A
\hat D^{-1/2}
$$

and:

- $H^{(l)}$ = node representations at layer $l$
- $W^{(l)}$ = learnable weight matrix
- $\sigma$ = activation function

At the first layer:

$$
H^{(0)} = X
$$

so:

$$
\boxed{
H^{(1)}
=
\sigma
\left(
S X W
\right)
}
$$

This is the equation we are going to implement.


In [6]:

# Node features
X = np.array([
    [1.0, 0.0],  # Alice
    [0.0, 1.0],  # Bob
    [1.0, 1.0],  # Charlie
    [0.0, 0.0],  # David
    [1.0, 0.0],  # Eve
])

# Learnable weight matrix
W = np.array([
    [0.5,  0.2],
    [-0.3, 0.8]
])

def relu(x):
    return np.maximum(0, x)

# GCN layer
H1 = relu(S @ X @ W)

print("H^(1):")
print(H1)


H^(1):
[[0.133 0.667]
 [0.067 0.333]
 [0.4   0.467]
 [0.067 0.333]
 [0.233 0.4  ]]



## Read the equation directly from the code

The implementation:

```python
H1 = relu(S @ X @ W)
```

corresponds exactly to:

$$
H^{(1)}
=
\sigma(SXW)
$$

and because:

$$
S =
\hat D^{-1/2}\hat A\hat D^{-1/2}
$$

we have:

$$
H^{(1)}
=
\sigma
\left(
\hat D^{-1/2}
\hat A
\hat D^{-1/2}
XW
\right)
$$

This is the central implementation goal of Tutorial 2.

---

# 9 — Understanding the order of operations

Matrix multiplication is associative, so we can think about the layer in stages.

### Stage 1 — Graph propagation

$$
SX
$$

The graph determines how information moves between nodes.

### Stage 2 — Feature transformation

$$
(SX)W
$$

The learnable matrix $W$ transforms the features.

### Stage 3 — Nonlinearity

$$
\sigma((SX)W)
$$

ReLU allows the network to learn nonlinear relationships.

So:

```text
X
│
├── Graph structure ──→ S
│
▼
S @ X
│
▼
(S @ X) @ W
│
▼
ReLU
│
▼
H¹
```



# 10 — What exactly is being learned?

This is an important distinction.

The graph structure is given to us:

$$
A
$$

The normalization is calculated from the graph:

$$
S =
\hat D^{-1/2}\hat A\hat D^{-1/2}
$$

But the weight matrix:

$$
W
$$

is **learnable**.

During training, the model changes $W$ so that the resulting node representations become useful for a task.

For example:

```text
Graph + features
       ↓
     GCN
       ↓
node representations
       ↓
classification
```

The GCN is therefore learning **how to transform and combine graph information**.



# 11 — A simple node-classification problem

Let's create a toy classification task.

Suppose every node belongs to one of two categories:

```text
Class 0 → [0, 1, 0, 1, ...]
Class 1 → [1, 0, 1, 0, ...]
```

Our goal is:

$$
\text{graph + node features}
\rightarrow
\text{node class}
$$

We can use a final linear layer:

$$
Z = H^{(1)}W_{\text{out}}
$$

where $Z$ contains one score for each class.

For this tutorial we will first do the forward pass manually.

Training will come in the next tutorial.


In [7]:

# Two-class output layer
W_out = np.array([
    [0.7, -0.2],
    [0.1,  0.9]
])

logits = H1 @ W_out

print("Class logits:")
print(logits)


Class logits:
[[0.16  0.573]
 [0.08  0.287]
 [0.327 0.34 ]
 [0.08  0.287]
 [0.203 0.313]]



For each node we now have two values:

```text
             Class 0     Class 1
Alice          ...         ...
Bob            ...         ...
Charlie        ...         ...
...
```

The larger value can be interpreted as the model's preferred class.

For a real classifier we would normally use a softmax and a loss function.

We will implement those properly once we introduce **training with PyTorch**.



# 12 — 🧪 Hands-on Exercise 3: Build the GCN layer yourself

Now write the GCN computation without copying the complete solution.

You already have:

```python
A
X
```

Your task is to calculate:

$$
H^{(1)}
=
\mathrm{ReLU}
\left(
\hat D^{-1/2}
\hat A
\hat D^{-1/2}
XW
\right)
$$

### Requirements

1. Create `A_hat`.
2. Calculate the node degrees.
3. Create `D_hat_inv_sqrt`.
4. Calculate `S`.
5. Create a weight matrix `W`.
6. Calculate `H1`.
7. Print `H1`.

Try to implement each mathematical step separately.

Do **not** write everything as one line initially.


In [ ]:

# YOUR CODE HERE

# Step 1: Add self-loops

# Step 2: Calculate degrees

# Step 3: Calculate D^(-1/2)

# Step 4: Calculate normalized adjacency S

# Step 5: Define W

# Step 6: Calculate H1

# Step 7: Print H1


In [8]:

# Solution

A_hat = A + np.eye(A.shape[0])

degrees_hat = A_hat.sum(axis=1)

D_hat_inv_sqrt = np.diag(
    1.0 / np.sqrt(degrees_hat)
)

S = D_hat_inv_sqrt @ A_hat @ D_hat_inv_sqrt

W = np.array([
    [0.5,  0.2],
    [-0.3, 0.8]
])

H1 = relu(S @ X @ W)

print(H1)


[[0.133 0.667]
 [0.067 0.333]
 [0.4   0.467]
 [0.067 0.333]
 [0.233 0.4  ]]



# 13 — 🧪 Hands-on Exercise 4: Compare raw and normalized aggregation

Now compare three approaches:

### Method 1

$$
AX
$$

### Method 2

$$
\hat AX
$$

### Method 3

$$
SX
$$

Calculate all three and inspect Alice's representation.

Then answer:

1. How does adding self-loops change the result?
2. How does normalization change the magnitude?
3. Why might normalization be useful when nodes have different degrees?


In [9]:

# YOUR CODE HERE

raw = A @ X
with_self_loops = A_hat @ X
normalized = S @ X

print("Alice:")
print("A @ X:             ", raw[0])
print("(A + I) @ X:       ", with_self_loops[0])
print("S @ X:             ", normalized[0])


Alice:
A @ X:              [1. 2.]
(A + I) @ X:        [2. 2.]
S @ X:              [0.667 0.667]



# 14 — 🧪 Hands-on Exercise 5: Add a second GCN layer

A major advantage of GNNs is that layers can be stacked.

We already have:

$$
H^{(1)}
=
\sigma(SXW^{(0)})
$$

Now calculate:

$$
H^{(2)}
=
\sigma(SH^{(1)}W^{(1)})
$$

Use a different weight matrix for the second layer.

### Question

After two graph-convolution layers, how far through the graph can information theoretically travel?

Think about the answer before reading the next section.


In [10]:

# YOUR CODE HERE

W1 = np.array([
    [0.5,  0.2],
    [-0.3, 0.8]
])

W2 = np.array([
    [0.4, -0.1],
    [0.6,  0.7]
])

# First layer
# H1 = ...

# Second layer
# H2 = ...

# print(H2)


In [11]:

# Solution

H1 = relu(S @ X @ W1)
H2 = relu(S @ H1 @ W2)

print("H1:")
print(H1)

print("\nH2:")
print(H2)


H1:
[[0.133 0.667]
 [0.067 0.333]
 [0.4   0.467]
 [0.067 0.333]
 [0.233 0.4  ]]

H2:
[[0.373 0.322]
 [0.302 0.302]
 [0.409 0.332]
 [0.262 0.237]
 [0.333 0.257]]



With one layer, information propagates roughly one graph hop.

With two layers, information can propagate roughly two hops.

```text
Layer 1:

A ← B


Layer 2:

A ← B ← C
```

Therefore:

$$
\boxed{
L\text{ GCN layers}
\rightarrow
\text{approximately }L\text{-hop receptive field}
}
$$

This idea becomes important later when discussing **over-smoothing** and GNN depth.



# 15 — The complete GCN in one function

Now that every component is understood, let's package it.



In [12]:

def gcn_layer(X, A, W):
    # Add self-loops
    A_hat = A + np.eye(A.shape[0])

    # Calculate degree
    degrees = A_hat.sum(axis=1)

    # D^(-1/2)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees))

    # Normalized adjacency
    S = D_inv_sqrt @ A_hat @ D_inv_sqrt

    # Graph convolution
    H = S @ X @ W

    # Nonlinearity
    H = np.maximum(0, H)

    return H


W = np.array([
    [0.5,  0.2],
    [-0.3, 0.8]
])

H = gcn_layer(X, A, W)

print(H)


[[0.133 0.667]
 [0.067 0.333]
 [0.4   0.467]
 [0.067 0.333]
 [0.233 0.4  ]]



Now we have compressed everything we learned into:

```python
H = gcn_layer(X, A, W)
```

But unlike a black-box library call, we know exactly what happens inside.

---

# 16 — The complete mathematical picture

Starting with:

$$
A
$$

we add self-loops:

$$
\hat A = A+I
$$

Then calculate:

$$
\hat D_{ii}
=
\sum_j\hat A_{ij}
$$

Then:

$$
\hat D^{-1/2}_{ii}
=
\frac{1}{\sqrt{\hat D_{ii}}}
$$

Then normalized propagation:

$$
S =
\hat D^{-1/2}
\hat A
\hat D^{-1/2}
$$

Finally:

$$
\boxed{
H^{(l+1)}
=
\sigma
\left(
S H^{(l)}W^{(l)}
\right)
}
$$

This is the basic GCN layer.

At the first layer:

$$
H^{(0)}=X
$$

so:

$$
\boxed{
H^{(1)}
=
\sigma
\left(
\hat D^{-1/2}
\hat A
\hat D^{-1/2}
XW^{(0)}
\right)
}
$$



# 17 — Why this is called a "graph convolution"

In an ordinary convolutional neural network, such as a CNN:

```text
image grid
    ↓
local neighbourhood
    ↓
weighted aggregation
    ↓
new feature
```

The neighbourhood is defined by the regular image grid.

For a graph:

```text
irregular graph
      ↓
graph neighbourhood
      ↓
weighted aggregation
      ↓
new node feature
```

The key difference is that graph connectivity can be **irregular**.

There is no requirement for nodes to lie on a rectangular grid.

This is one reason GNNs are useful for:

- molecules
- social networks
- meshes
- road networks
- physical systems
- particle simulations



# 18 — Engineering connection: from GCN to CFD

This is where the idea becomes particularly relevant to scientific machine learning.

Consider a CFD mesh.

Each cell can be represented as a node:

$$
i \rightarrow \text{cell }i
$$

Neighbouring cells become graph edges:

$$
(i,j)\in E
$$

Node features might contain:

$$
x_i =
[
p_i,\,
u_i,\,
v_i,\,
w_i,\,
T_i,\,
\rho_i
]
$$

Then a GNN can perform:

$$
X
\rightarrow
\text{graph propagation}
\rightarrow
H
\rightarrow
\text{prediction}
$$

Conceptually, this resembles a learned form of local information propagation.

This does **not** mean a GCN automatically solves the governing equations.

Instead, it gives us a machine-learning architecture whose connectivity follows the computational domain.

We will explore this properly in the later scientific-ML tutorials.



# 19 — Final Challenge 🚀

Build a complete **two-layer GCN forward pass** for your own graph.

### Requirements

Create:

- at least 6 nodes
- at least 8 edges
- 3 input features per node
- a first GCN layer with 4 hidden features
- a second GCN layer with 2 output features

Your pipeline should be:

$$
X
\rightarrow
H^{(1)}
\rightarrow
H^{(2)}
$$

using:

$$
H^{(1)}
=
\mathrm{ReLU}(SXW_1)
$$

and:

$$
H^{(2)}
=
SX^{(1)}W_2
$$

where:

$$
X^{(1)}=H^{(1)}
$$

### Bonus

Implement the normalized adjacency calculation as a separate function:

```python
def normalized_adjacency(A):
    ...
```

and then use:

```python
S = normalized_adjacency(A)
```

This is a good point to stop looking at the previous examples and try the entire pipeline yourself.



---

# 20 — Key vocabulary

```text
Degree              Number of neighbours of a node

Degree matrix D     Diagonal matrix containing node degrees

Self-loop           Connection from a node to itself

A_hat               Adjacency matrix with self-loops

D_hat               Degree matrix of A_hat

Normalization       Scaling graph propagation using node degrees

Propagation matrix  D^(-1/2) A D^(-1/2)

GCN                 Graph Convolutional Network

GCN layer            Graph propagation + learnable transformation + activation

Receptive field     Part of the graph that can influence a node

Logits               Raw outputs before softmax
```

---

# 21 — What you should understand now

You should be able to explain:

### 1. Why self-loops are added

$$
\hat A=A+I
$$

They allow each node to retain information about itself.

### 2. Why we need a degree matrix

$$
D_{ii}=d_i
$$

It tells us how connected each node is.

### 3. Why we normalize

$$
\hat D^{-1/2}\hat A\hat D^{-1/2}
$$

Normalization prevents graph propagation from being dominated simply by node degree.

### 4. What the learnable weights do

$$
W
$$

They transform the propagated node features.

### 5. What a GCN layer does

$$
\boxed{
H^{(l+1)}
=
\sigma
\left(
\hat D^{-1/2}
\hat A
\hat D^{-1/2}
H^{(l)}
W^{(l)}
\right)
}
$$

---

# 22 — Next tutorial

We now understand the **forward pass** of a GCN.

But the weights $W$ were simply chosen by us.

The next question is:

> **How does the network learn the correct weights?**

That leads to:

# Tutorial 3 — Training a GCN with PyTorch

We will introduce:

- tensors
- `nn.Module`
- learnable parameters
- forward pass
- softmax
- cross-entropy loss
- backpropagation
- gradient descent
- training loops
- node classification

And we will build the first **actually trainable GCN**.

Only after that will we introduce **PyTorch Geometric**, where we can compare our implementation with `GCNConv`.

---

## 🎯 The progression so far

```text
Tutorial 1
Graph
  ↓
A + X
  ↓
Neighbour aggregation
  ↓
Message passing

Tutorial 2
A + I
  ↓
Degree matrix
  ↓
Normalization
  ↓
GCN
  ↓
Forward pass

Tutorial 3
GCN
  ↓
Loss
  ↓
Backpropagation
  ↓
Training
  ↓
Learned GNN
```

The important thing is that we are building the intuition **before hiding it behind a library**.


---
## References
- Defferrard, M., Bresson, X., & Vandergheynst, P. (2016). Convolutional Neural Networks on Graphs with Fast Localized Spectral Filtering. NeurIPS. https://arxiv.org/abs/1606.09375
- Kipf, T. N., & Welling, M. (2017). Semi-Supervised Classification with Graph Convolutional Networks. ICLR. https://arxiv.org/abs/1609.02907

-- Rohit Raj $\cdot$ HalfQubiX